# Naive Bayes Baseline Results

This notebook trains a TfidfVectorizer + MultinomialNB baseline on Mental-Health-Twitter.csv and displays the evaluation metrics in a Markdown table.

In [1]:
import pandas as pd
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

In [2]:
DATA_PATH = 'Mental-Health-Twitter.csv'
TEXT_COLUMN = 'post_text'
LABEL_COLUMN = 'label'
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [3]:
df = pd.read_csv(DATA_PATH)
dataset = df[[TEXT_COLUMN, LABEL_COLUMN]].dropna().copy()
dataset[TEXT_COLUMN] = dataset[TEXT_COLUMN].astype(str).str.strip()
dataset = dataset[dataset[TEXT_COLUMN] != '']
labels = dataset[LABEL_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    dataset[TEXT_COLUMN],
    labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)

In [4]:
model = Pipeline(
    steps=[
        (
            'tfidf',
            TfidfVectorizer(
                lowercase=True,
                stop_words='english',
                ngram_range=(1, 2),
                min_df=1,
            ),
        ),
        ('classifier', MultinomialNB()),
    ]
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [5]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)
cm = confusion_matrix(y_test, predictions)

results_df = pd.DataFrame(
    {
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
        'Value': [accuracy, precision, recall, f1],
    }
)
results_df['Value'] = results_df['Value'].map(lambda value: f'{value:.4f}')
results_df

,Metric,Value
0,Accuracy,0.8875
1,Precision,0.8676
2,Recall,0.9145
3,F1-score,0.8905


In [6]:
markdown_lines = [
    '| Metric | Value |',
    '|---|---:|',
]
for _, row in results_df.iterrows():
    markdown_lines.append(f"| {row['Metric']} | {row['Value']} |")

display(Markdown('## Evaluation Metrics\n\n' + '\n'.join(markdown_lines)))
display(Markdown('## Confusion Matrix'))
display(pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Pred 0', 'Pred 1']))

## Evaluation Metrics

| Metric | Value |
|---|---:|
| Accuracy | 0.8875 |
| Precision | 0.8676 |
| Recall | 0.9145 |
| F1-score | 0.8905 |

## Confusion Matrix

,Pred 0,Pred 1
Actual 0,1721,279
Actual 1,171,1829
